# Pipeline de Limpeza e Classificação — LinkedIn Jobs

Pipeline sequencial:
1. Carrega dados do banco (jobs × skills)
2. Normaliza cargo → `position_group`
3. Detecta senioridade → `level` (explode vagas multi-nível)
4. Padroniza skills → `skill_grouped`
5. Classifica tipo da skill → `tipo_skill` (tecnica / soft)
6. Exporta `data/skills_standardized.csv`

## 1. Imports e Conexão com o Banco

In [ ]:
import re
import sys
import pandas as pd
from pathlib import Path
from unidecode import unidecode
from sqlalchemy import create_engine

# Coloca a raiz do repo no path para importar módulos comuns
sys.path.insert(0, str(Path('..').resolve()))

DB_HOST = "teste"
DB_PORT = 0000
DB_NAME = "db_name"
DB_USER = "db_user"
DB_PASSWORD = "db_password"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

df = pd.read_sql(
    """
    SELECT j.job_id, j.title, j.company, j.city, j.work_mode, js.skill
    FROM jobs j
    INNER JOIN job_skills js ON j.job_id = js.job_id
    """,
    engine
)

print(f"Linhas carregadas: {len(df):,}")
print(f"Vagas únicas: {df['job_id'].nunique():,}")
df.head()

Linhas carregadas: 78,643
Vagas únicas: 8,214


,job_id,title,company,city,work_mode,skill
0,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Localiza&Co,Brasil,remoto,AWS
1,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Localiza&Co,Brasil,remoto,Agile
2,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Localiza&Co,Brasil,remoto,Airflow
3,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Localiza&Co,Brasil,remoto,Azure
4,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Localiza&Co,Brasil,remoto,Big Data


## 2. Classificação de Cargo (`position_group`)

In [9]:
CATEGORIAS = {
    "Data Analyst", "Data Engineer", "BI Analyst", "Data Intern",
    "Analytics Engineer", "Data Scientist", "Data Architect",
    "Data Leadership", "Data Product", "Data Governance",
    "Data Talent Pool", "Not Data Role",
}


def normalize_position(position: str) -> str:
    """Agrupa o título da vaga em uma das 12 categorias canônicas."""
    if str(position) in CATEGORIAS:
        return str(position)

    p = unidecode(str(position).lower().strip())
    # Remove pontuação que quebra o 'in' (parênteses, barras, pipes, etc.)
    p = re.sub(r"[()/|.,{}\\'\"\[\]]", " ", p)
    p = re.sub(r"\s+", " ", p).strip()

    # =========================
    # BANCO DE TALENTOS
    # =========================
    if "banco de talentos" in p:
        return "Data Talent Pool"

    # =========================
    # NÃO É DADOS (antes das regras de dados para não roubar match)
    # =========================
    if any(x in p for x in [
        "analista de ia", "consultor ti - ia", "consultor ti ia",
        "analista de monitoramento", "prevencao de perdas",
        "analista de sustentabilidade", "analista administrativo",
        "relacoes institucionais", "analista de melhoria continua",
        "analista de gestao", "analista sistema de gestao",
        "analista de controle e gestao", "analista de cadastro",
        "analista nto", "engenheiro de sistemas", "analista de sistemas jr",
        "coletor de dados", "bolsista de inovacao", "assistente de qualidade",
        "analista de atrativos", "analista jr de projetos de engenharia",
        "servicos externos",
    ]):
        return "Not Data Role"

    # =========================
    # DATA LEADERSHIP
    # =========================
    if any(x in p for x in [
        "gerente de dados", "lead dados", "lead de dados", "data lead",
        "head of data", "tech lead", "data engineering lead",
        "data analytics coordinator", "data science coordinator",
        "coordenador de dados", "coordenador de dados de mercado",
        "engineering manager", "data leadership",
    ]):
        return "Data Leadership"

    # =========================
    # DATA PRODUCT
    # =========================
    if any(x in p for x in [
        "data product owner", "data product manager", "data products specialist",
    ]):
        return "Data Product"

    # =========================
    # DATA ARCHITECT
    # =========================
    if any(x in p for x in [
        "data architect", "cloud data architect", "arquiteto de dados",
    ]):
        return "Data Architect"

    # =========================
    # DATA GOVERNANCE
    # =========================
    if any(x in p for x in [
        "governanca de dados", "data governance",
        "commercial data governance analyst",
    ]):
        return "Data Governance"

    # =========================
    # DATA SCIENTIST
    # =========================
    if any(x in p for x in [
        "data scientist", "staff data scientist", "data science",
        "cientista de dados", "cientista dados", "ciencia de dados",
        "machine learning", "mlops", "artificial intelligence",
        "inteligencia artificial", "genai", "generative ai", "ai engineer",
        "especialista em ciencia de dados", "forward deployed data scientist",
        "python engineer - machine learning focus", "analista data science dados",
    ]):
        return "Data Scientist"

    # =========================
    # ANALYTICS ENGINEER
    # =========================
    if any(x in p for x in [
        "analytics engineer", "analytics engineer iii", "analytics engineer junior",
        "staff analytics engineer", "senior analytics engineer",
        "data analytics engineer", "data analyst engineer",
        "engenharia de analytics", "engenheiro de analytics",
        "engenharia de dados & analytics", "engenharia de dados e analytics",
        "plataforma de analytics",
    ]):
        return "Analytics Engineer"

    # =========================
    # DATA ENGINEER
    # =========================
    if any(x in p for x in [
        "data engineer", "senior data engineer", "junior data engineer",
        "staff data engineer", "data engineer specialist",
        "data engineer consultant", "data engineer trainee",
        "engenheiro de dados", "engenheira de dados", "engenheiro a de dados",
        "engenharia de dados", "eng de dados", "engenheiro de banco de dados",
        "engenheiro de dados azure", "engenheiro de dados azure databricks",
        "engenheiro solucoes dados", "data solutions",
        "engenheiro de confiabilidade de banco de dados", "etl engineer",
        "etl data engineer", "desenvolvimento de etl", "dados e etl",
        "snowflake engineer", "data platform engineer",
        "senior data platform engineer", "plataforma de dados",
        "dataops engineer", "database reliability engineer",
        "senior database reliability engineer", "big data engineer",
        "data quality engineer", "data solutions engineer",
        "data management engineer", "data engineering specialist",
        "especialista engenharia de dados", "especialista engenharia dados",
        "especialista em engenharia de dados", "suporte de banco de dados",
        "suporte de banco dados", "dev banco dados", "dev banco de dados",
        "databricks",
    ]):
        return "Data Engineer"

    # =========================
    # BI / ANALYTICS
    # =========================
    if any(x in p for x in [
        "business intelligence", "business intelligence analyst",
        "business intelligence specialist", " bi ", "de bi", "a bi",
        "power bi", "desenvolvedor de bi", "inteligencia comercial",
        "inteligencia de mercado", "inteligencia de negocios",
        "inteligencia competitiva", "inteligencia operacional",
        "people analytics", "data analytics", "data & analytics",
        "data visualization", "data visualization specialist",
        "consultor dados business intelligence", "martech analyst",
        "analytics intern", "specialista em analytics", "indicadores",
    ]):
        return "BI Analyst"

    # =========================
    # DATA ANALYST
    # =========================
    if any(x in p for x in [
        "analista de dados", "analista dados", "analistas de dados",
        "data analyst", "data & ai analyst", "finance data analyst",
        "fraud data analyst", "industrial data analyst", "business data analyst",
        "hr data analyst", "etl data analyst", "datahub analyst",
        "data migration analyst", "data automation analyst",
        "data development analyst", "data intelligence analyst",
        "market intelligence analyst", "senior data analyst", "dados junior",
        "dados pleno", "dados senior", "pleno de dados", "senior de dados",
        "junior de dados", "planejamento dados", "planejamento e dados",
        "operacoes de dados", "projetos de dados", "projetos e dados",
        "validacao de dados", "migracao de dados", "inteligencia de dados",
        "especialista dados", "especialista em dados", "especialista de dados",
        "finance data specialist", "analista de modelagem de dados",
        "analista de base de dados", "analista de governanca de dados",
        "analista de dados dba", "coordenador de dados de mercado",
        "ti e dados", "logistico de dados", "analise de dados",
        "foco em dados", "data & ai",
    ]):
        return "Data Analyst"

    # =========================
    # INTERNS
    # =========================
    if any(x in p for x in ["intern", "estagi", "stagiaire", "trainee"]):
        return "Data Intern"

    return "Not Data Role"


df["position_group"] = df["title"].apply(normalize_position)

print("\nDistribuição por cargo:")
print(
    df.groupby("position_group")["job_id"]
    .nunique()
    .sort_values(ascending=False)
    .to_frame(name="vagas")
)


Distribuição por cargo:
                    vagas
position_group           
Not Data Role        3048
Data Analyst         1483
Data Engineer        1447
Data Scientist       1086
BI Analyst            584
Data Intern           170
Data Talent Pool      132
Analytics Engineer    131
Data Leadership        49
Data Governance        45
Data Architect         36
Data Product            3


## 3. Detecção de Senioridade (`level`) + Explode

Uma vaga pode ter múltiplos níveis (ex: "Pleno/Sênior"). Nesses casos a linha
é duplicada para cada nível detectado, permitindo contabilizar nos dois grupos.

In [10]:
_LEVEL_PATTERNS = [
    ("estagio",  r"\bestagi|\btrainee\b|\bintern\b|interns?hip|\bstagiaire\b"),
    ("junior",   r"\bjr\b|\bjunior\b|\bjr\.|j[uú]nior"),
    ("pleno",    r"\bpl\b|\bpl\.|\bpleno\b|\bplena\b|midlevel|mid-level|mid level|\bintermediate\b|intermediari[oa]"),
    ("senior",   r"\bsr\b|\bsr\.|s[eê]nior\b|\bstaff\b"),
]


def detect_seniority(title: str) -> list:
    """Retorna lista de níveis detectados no título.
    
    Sem marcador explícito retorna ['nao_especificado'].
    Retorna lista para permitir explode de vagas multi-nível.
    """
    p = unidecode(str(title).lower())
    p = re.sub(r"[/|]", " ", p)  # "pleno/senior" → "pleno senior"
    found = [lvl for lvl, pat in _LEVEL_PATTERNS if re.search(pat, p)]
    return found or ["nao_especificado"]


# Testes rápidos
test_titles = [
    "ANALISTA DE DADOS PL",
    "Engenheiro de Dados Jr.",
    "Engenheiro de Dados Pleno/Sênior",
    "Estágio - Dados",
    "Midlevel Analytics Engineer",
    "Analista de Dados",
    "Engenharia de Dados & Analytics Senior",
]
print("Testes detect_seniority:")
for t in test_titles:
    print(f"  {t:45s} → {detect_seniority(t)}")

# Aplica e explode — vagas "Pleno/Sênior" viram 2 linhas
df["level"] = df["title"].apply(detect_seniority)
df = df.explode("level", ignore_index=True)

print(f"\nLinhas após explode: {len(df):,}")
print("\nDistribuição de senioridade (vagas únicas por nível):")
labels = {"estagio": "Estágio", "junior": "Júnior", "pleno": "Pleno",
          "senior": "Sênior", "nao_especificado": "Não especificado"}
lvl_counts = df.groupby("level")["job_id"].nunique().sort_values(ascending=False)
for k, v in lvl_counts.items():
    print(f"  {labels.get(k, k):20s}: {v:5d}")

Testes detect_seniority:
  ANALISTA DE DADOS PL                          → ['pleno']
  Engenheiro de Dados Jr.                       → ['junior']
  Engenheiro de Dados Pleno/Sênior              → ['pleno', 'senior']
  Estágio - Dados                               → ['estagio']
  Midlevel Analytics Engineer                   → ['pleno']
  Analista de Dados                             → ['nao_especificado']
  Engenharia de Dados & Analytics Senior        → ['senior']

Linhas após explode: 79,726

Distribuição de senioridade (vagas únicas por nível):
  Não especificado    :  4546
  Sênior              :  1949
  Pleno               :   903
  Júnior              :   634
  Estágio             :   283


## 4. Padronização de Skills (`skill_grouped`)

Unifica variações de nome (ex: "AWS", "Amazon Web Services", "AWS (Amazon)" → "AWS").

In [11]:
_EXACT_SKILL_MAP = {
    # Cloud
    "aws": "AWS",
    "amazon web services": "AWS",
    "amazon web services (aws)": "AWS",
    "gcp": "Google Cloud Platform",
    "google cloud": "Google Cloud Platform",
    "google cloud platform (gcp)": "Google Cloud Platform",
    # Azure
    "azure": "Azure",
    "microsoft azure": "Azure",
    "cosmosdb": "Azure Cosmos DB",
    "azure cosmos db": "Azure Cosmos DB",
    "cosmos db": "Azure Cosmos DB",
    # BI
    "powerbi": "Power BI",
    "power bi": "Power BI",
    "microsoft power bi": "Power BI",
    # Excel
    "excel": "Microsoft Excel",
    "microsoft excel": "Microsoft Excel",
    "excel avancado": "Microsoft Excel",
    "excel intermediario": "Microsoft Excel",
    # Bancos
    "postgres": "PostgreSQL",
    "postgresql": "PostgreSQL",
    "ms sql server": "SQL Server",
    "sql server": "SQL Server",
    "mssql": "SQL Server",
    "microsoft sql server": "SQL Server",
    # Linguagens
    "phyton": "Python",
    "python": "Python",
    "pyspark": "PySpark",
    "spark": "Apache Spark",
    "apache spark": "Apache Spark",
    # dbt
    "dbt (data build tool)": "dbt",
    "dbt core": "dbt",
    "dbt": "dbt",
    # LLMs
    "llms": "LLM",
    "llm": "LLM",
    "large language models (llms)": "LLM",
    "large language models": "LLM",
    # Soft skills (manter grafia correta com acentos)
    "comunicacao": "Comunicação",
    "organizacao": "Organização",
    "lideranca": "Liderança",
    "resolucao de problemas": "Resolução de Problemas",
    "pensamento analitico": "Pensamento Analítico",
    "visao estrategica": "Visão Estratégica",
}


def standardize_skill(skill: str) -> str:
    """Unifica variações de nome de skill."""
    s = unidecode(str(skill).lower().strip())

    # 1. Mapeamento exato
    if s in _EXACT_SKILL_MAP:
        return _EXACT_SKILL_MAP[s]

    # 2. Mapeamento por substring (mais específico primeiro)
    if "cosmos" in s:
        return "Azure Cosmos DB"
    if "databricks" in s:
        return "Databricks"
    if "bigquery" in s or "big query" in s:
        return "Google BigQuery"
    if "machine learning" in s:
        return "Machine Learning"
    if "comunicacao" in s:
        return "Comunicação"

    # 3. Fallback: Title Case da string original
    return str(skill).strip().title()


df["skill_grouped"] = df["skill"].apply(standardize_skill)

print("Top 20 skills após padronização:")
print(
    df.groupby("skill_grouped")["job_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(20)
    .to_frame(name="vagas")
)

Top 20 skills após padronização:
                       vagas
skill_grouped               
Sql                     4612
Python                  4388
Data Pipelines          2819
Power BI                2631
Ingles                  2339
Microsoft Office        1880
Microsoft Excel         1865
AWS                     1786
Etl                     1672
Apis                    1651
Git                     1644
Machine Learning        1611
Data Modeling           1537
Workflow                1407
Azure                   1384
Agile                   1309
Databricks              1184
Tableau                 1160
Ci/Cd                   1157
Google Cloud Platform   1080


## 5. Classificação do Tipo de Skill (`tipo_skill`)

In [12]:
SOFT_SKILLS_NORM = {
    unidecode(s).lower() for s in {
        "Comunicação", "Trabalho em Equipe", "Proatividade",
        "Resolução de Problemas", "Pensamento Analítico", "Organização",
        "Autonomia", "Liderança", "Curiosidade", "Storytelling",
        "Visão Estratégica",
    }
}


def classify_skill_type(skill: str) -> str:
    """Retorna 'soft' ou 'tecnica' com base na skill padronizada."""
    return "soft" if unidecode(str(skill)).lower() in SOFT_SKILLS_NORM else "tecnica"


df["tipo_skill"] = df["skill_grouped"].apply(classify_skill_type)

print("Distribuição por tipo de skill:")
print(df["tipo_skill"].value_counts())
print("\nAmostra do DataFrame final:")
df[["job_id", "title", "position_group", "level", "skill", "skill_grouped", "tipo_skill"]].head(10)

Distribuição por tipo de skill:
tipo_skill
tecnica    79726
Name: count, dtype: int64

Amostra do DataFrame final:


,job_id,title,position_group,level,skill,skill_grouped,tipo_skill
0,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Data Engineer,pleno,AWS,AWS,tecnica
1,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Data Engineer,pleno,Agile,Agile,tecnica
2,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Data Engineer,pleno,Airflow,Airflow,tecnica
3,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Data Engineer,pleno,Azure,Azure,tecnica
4,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Data Engineer,pleno,Big Data,Big Data,tecnica
5,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Data Engineer,pleno,Data Governance,Data Governance,tecnica
6,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Data Engineer,pleno,Data Modeling,Data Modeling,tecnica
7,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Data Engineer,pleno,Data Pipelines,Data Pipelines,tecnica
8,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Data Engineer,pleno,Data Warehouse,Data Warehouse,tecnica
9,4419602799,ENGENHEIRO DE DADOS PLENO | REMOTO,Data Engineer,pleno,Dimensional Modeling,Dimensional Modeling,tecnica


## 6. Verificação e Inspeção Final

In [13]:
print("=== Resumo do Dataset ===")
print(f"  Total linhas        : {len(df):>8,}")
print(f"  Vagas únicas        : {df['job_id'].nunique():>8,}")
print(f"  Skills únicas       : {df['skill_grouped'].nunique():>8,}")
print(f"  Cargos identificados: {df[df['position_group'] != 'Not Data Role']['job_id'].nunique():>8,}")

print("\n=== Cargos (vagas únicas) ===")
EXCLUDED = {"Not Data Role", "Data Talent Pool"}
print(
    df.groupby("position_group")["job_id"]
    .nunique()
    .sort_values(ascending=False)
    .to_frame(name="vagas")
)

print("\n=== Senioridade (vagas únicas, sem Not Data Role) ===")
df_data = df[~df["position_group"].isin(EXCLUDED)]
labels = {"estagio": "Estágio", "junior": "Júnior", "pleno": "Pleno",
          "senior": "Sênior", "nao_especificado": "Não especificado"}
lvl = df_data.groupby("level")["job_id"].nunique().sort_values(ascending=False)
total = df_data["job_id"].nunique()
for k, v in lvl.items():
    print(f"  {labels.get(k, k):20s}: {v:5d}  ({100*v/total:.1f}%)")

=== Resumo do Dataset ===
  Total linhas        :   79,726
  Vagas únicas        :    8,214
  Skills únicas       :      230
  Cargos identificados:    5,166

=== Cargos (vagas únicas) ===
                    vagas
position_group           
Not Data Role        3048
Data Analyst         1483
Data Engineer        1447
Data Scientist       1086
BI Analyst            584
Data Intern           170
Data Talent Pool      132
Analytics Engineer    131
Data Leadership        49
Data Governance        45
Data Architect         36
Data Product            3

=== Senioridade (vagas únicas, sem Not Data Role) ===
  Não especificado    :  2931  (58.2%)
  Sênior              :  1132  (22.5%)
  Pleno               :   498  (9.9%)
  Estágio             :   273  (5.4%)
  Júnior              :   237  (4.7%)


## 7. Exportar CSV

O arquivo `data/skills_standardized.csv` será lido por `analysis/dashboard_from_csv.py`
para gerar os slides do carrossel.

In [14]:
OUTPUT_PATH = Path("..") / "data" / "skills_standardized.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Colunas relevantes para o dashboard
COLS = [
    "job_id", "title", "company", "city", "work_mode",
    "position_group", "level",
    "skill", "skill_grouped", "tipo_skill",
]

df[COLS].to_csv(OUTPUT_PATH, index=False)
print(f"✓ CSV exportado → {OUTPUT_PATH.resolve()}")
print(f"  {len(df):,} linhas | {df['job_id'].nunique():,} vagas únicas")

✓ CSV exportado → /home/victor/projects/linkedin-job-analysis/data/skills_standardized.csv
  79,726 linhas | 8,214 vagas únicas
